# Lab 06 External V2 — 05 Register Shared Tables

**Dataset:** Synthea Healthcare  
**Architecture:** External Delta tables  
**Compute:** Databricks Serverless compatible

## Purpose

Validate and, when needed, register all core Lab 06 external Delta locations
in the target Unity Catalog schema. Existing tables are never moved to a different
location automatically.

> Serverless compatibility rule: this notebook does **not** call
> `REFRESH TABLE`, `CACHE TABLE`, `UNCACHE TABLE`, or Spark cache-refresh APIs.


## 1. Runtime context

In [0]:
def ensure_text_widget(name: str, default: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(
    name: str,
    default: str,
    choices: list[str],
    label: str,
) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


ensure_text_widget("catalog", "dbr_dev", "01 Catalog")
ensure_text_widget("source_schema", "parvinbadalov", "02 Source schema")
ensure_text_widget(
    "source_volume_name",
    "lab06_gold_analytics",
    "03 Source volume",
)
ensure_text_widget(
    "target_schema",
    "parvinbadalov_lab06_ext",
    "04 Target schema",
)
ensure_text_widget(
    "external_gold_root",
    "AUTO",
    "05 External Gold root",
)
ensure_dropdown_widget(
    "run_validation",
    "true",
    ["true", "false"],
    "06 Run validation",
)

catalog = dbutils.widgets.get("catalog").strip()
source_schema = dbutils.widgets.get("source_schema").strip()
source_volume_name = dbutils.widgets.get("source_volume_name").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
external_gold_root = dbutils.widgets.get("external_gold_root").strip().rstrip("/")
run_validation = (
    dbutils.widgets.get("run_validation").strip().lower() == "true"
)

source_volume_path = (
    f"/Volumes/{catalog}/{source_schema}/{source_volume_name}"
)
source_csv_path = f"{source_volume_path}/source/csv"
reference_path = f"{source_volume_path}/reference"
target_schema_fqn = f"{catalog}.{target_schema}"

# Manual-run support:
# 05/06 are outside the recurring dev-runner chain, so AUTO derives the
# external Gold root from the already-created external dim_date table.
if (
    not external_gold_root
    or external_gold_root.upper() == "AUTO"
    or external_gold_root == "REPLACE_WITH_EXTERNAL_GOLD_ROOT"
):
    dim_date_table = f"{target_schema_fqn}.dim_date"

    if not spark.catalog.tableExists(dim_date_table):
        raise RuntimeError(
            "external_gold_root could not be auto-detected because "
            f"{dim_date_table} does not exist. Run 01_dimensions first, "
            "or enter the ABFSS external Gold root manually."
        )

    dim_detail = spark.sql(
        f"DESCRIBE DETAIL {dim_date_table}"
    ).first()

    dim_location = str(
        dim_detail.asDict().get("location", "")
    ).rstrip("/")

    suffix = "/dim_date"

    if not dim_location.lower().endswith(suffix):
        raise RuntimeError(
            "Could not derive external_gold_root from dim_date location: "
            f"{dim_location}"
        )

    external_gold_root = dim_location[:-len(suffix)]

if not external_gold_root.lower().startswith("abfss://"):
    raise ValueError(
        "external_gold_root must resolve to an abfss:// Azure storage path."
    )

print(f"Catalog            : {catalog}")
print(f"Source schema      : {source_schema}")
print(f"Source volume      : {source_volume_name}")
print(f"Target schema      : {target_schema}")
print(f"External Gold root : {external_gold_root}")
print(f"Run validation     : {run_validation}")

## 2. Register/validate the external Gold objects

In [0]:
import sys
from pathlib import Path

from pyspark.sql import functions as F

current_dir = Path.cwd()
lab_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.external_tables import (
    normalize_location,
    overwrite_external_delta,
    registered_table_location,
    register_external_delta_table,
    validate_registered_location,
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema_fqn}")

print("Serverless-safe external-table helpers loaded.")

OBJECT_NAMES = [
    "dim_date",
    "dim_patient",
    "dim_provider",
    "dim_organization",
    "dim_payer",
    "dim_condition",
    "fact_encounters",
    "fact_conditions",
    "agg_daily_encounters",
    "agg_organization_performance",
    "agg_payer_performance",
    "agg_condition_summary",
]

rows = []
failures = []

for object_name in OBJECT_NAMES:
    table_name = f"{target_schema_fqn}.{object_name}"
    location = f"{external_gold_root}/{object_name}"

    # Verify the path contains Delta data before registration.
    try:
        path_rows = spark.read.format("delta").load(location).limit(1).count()
        delta_path_ok = True
    except Exception as exc:
        delta_path_ok = False
        failures.append(f"{object_name}: unreadable Delta location")
        rows.append((object_name, table_name, location, None, "FAIL"))
        continue

    try:
        register_external_delta_table(
            spark,
            table_name,
            location,
        )
        actual = registered_table_location(spark, table_name)
        status = (
            "PASS"
            if normalize_location(actual) == normalize_location(location)
            else "FAIL"
        )
    except Exception as exc:
        actual = registered_table_location(spark, table_name)
        status = "FAIL"
        failures.append(f"{object_name}: {type(exc).__name__}: {exc}")

    rows.append((object_name, table_name, location, actual, status))

registration_df = spark.createDataFrame(
    rows,
    ["object_name","table_name","expected_location","actual_location","status"],
)

display(registration_df)

if run_validation and failures:
    raise RuntimeError(
        "Shared external-table registration failed: "
        + " | ".join(failures)
    )

print("LAB 06 EXTERNAL V2 — SHARED TABLE REGISTRATION COMPLETE")
print("Serverless compatibility: PASS")
print("Unsupported cache-management commands: 0")